In [1]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

/NFSHOME/adangelo/miniconda3/envs/pytorch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [3]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,1677965479 | INFO | 1226080 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc


�u{��,1677965572 | INFO | 1226080 - Setting seeds to: 0
!,1677965822 | INFO | 1226080 - REMOVAL TYPE SET AS edge
,1677965823 | INFO | 1226080 - Caching System: False.
�{��,1677965947 | INFO | 1226080 - Instantiating: torch_geometric.datasets.Planetoid
�|�/�,1677966398 | INFO | 1226080 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
�f"1�,1677966399 | INFO | 1226080 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
�{��,1677966822 | INFO | 1226080 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0���,1677966966 | INFO | 1226080 - ['all', 'all_shuffled', '-']
�{��,1677966967 | INFO | 1226080 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
0�/�,1677966978 | INFO | 1

In [4]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

�{��,1677984506 | INFO | 1226080 - Instantiating: erasure.model.graphs.GCN.GCN
�{��,1677984722 | INFO | 1226080 - Instantiating: torch.optim.Adam
�{��,1677984723 | INFO | 1226080 - Instantiating: torch.nn.CrossEntropyLoss
��t,1677985755 | INFO | 1226080 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
��t,1677985840 | INFO | 1226080 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
��t,1677985897 | INFO | 1226080 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
��t,1677985953 | INFO | 1226080 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
��t,1677986011 | INFO | 1226080 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
��t,1677986066 | INFO | 1226080 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
��t,1677986121 | INFO | 1226080 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
��t,1677986176 | INFO | 1226080 - epoch = 7 ---> loss = 1.4527	 accuracy = 0.7378
��t,1677986231 | INFO | 1226080 - epoch = 8 ---> loss = 1.3981	 accuracy = 0.7432
��t,1677986285 | INFO | 1226080 - 

In [5]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [6]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [7]:
import torch
print(torch.version.cuda)

12.6


In [8]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [9]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [10]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

data manager <erasure.data.datasets.DatasetManager.DatasetManager object at 0x7fc0120c5820>
�|�/�,1677990031 | INFO | 1226080 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
�{��,1677990222 | INFO | 1226080 - Instantiating: torch.optim.Adam
�|�/�,1677990226 | INFO | 1226080 - Created Configurable: erasure.unlearners.graph_unlearners.UNSIR.UNSIR
�{��,1677990288 | INFO | 1226080 - Instantiating: torch.optim.Adam
�|�/�,1677990289 | INFO | 1226080 - Created Configurable: erasure.unlearners.graph_unlearners.SelectiveSynapticDampening.SelectiveSynapticDampening
�|�/�,1677990332 | INFO | 1226080 - Created Configurable: erasure.unlearners.composite.Identity


/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/GraphUnlearner.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(self.labels)


In [11]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [12]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [13]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [14]:
print(len( data_manager.partitions['forget']) )

1820


In [15]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

�|�/�,1677990975 | INFO | 1226080 - Created Configurable: erasure.evaluations.running.RunTime
�|�/�,1677991069 | INFO | 1226080 - Created Configurable: erasure.evaluations.measures.SaveValues
�|�/�,1677991070 | INFO | 1226080 - Created Configurable: erasure.evaluations.manager.Evaluator
pJ�e�,1677991070 | INFO | 1226080 - ####		 Evaluating Unlearner GoldModelGraph 		####
�}y/�,1677991106 | INFO | 1226080 - Unlearning copyed predictor: <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7fc1c41c0250>
�{��,1677991107 | INFO | 1226080 - Instantiating: erasure.model.graphs.GCN.GCN
�{��,1677991116 | INFO | 1226080 - Instantiating: torch.optim.Adam
�{��,1677991116 | INFO | 1226080 - Instantiating: torch.nn.CrossEntropyLoss
��t,1677991173 | INFO | 1226080 - epoch = 0 ---> loss = 1.7887	 accuracy = 0.1871
��t,1677991225 | INFO | 1226080 - epoch = 1 ---> loss = 1.7329	 accuracy = 0.3812
��t,1677991277 | INFO | 1226080 - epoch = 2 ---> loss = 1.6814	 accuracy = 0.5027
��t,16779